# YOLO nodule detector: train on Kaggle (new run, or continue one)

Jairus

Continues a training run from its **last saved epoch** (e.g. one started on Google Colab), or, if there is no run yet for the model and image size in cell 1, **starts a new one** with the same settings as the earlier runs. When continuing, nothing about the model, data or settings changes: the checkpoint carries every training setting with it.

**To train a different model** (e.g. YOLO11s as the comparison model), just change `MODEL` in cell 1. The first run starts it; later runs continue it.

### One-time setup
1. **Kaggle:** create an account, then Settings → **verify your phone number** (needed for GPU and internet).
2. **Only when continuing a run from Colab:** in Google Drive, right-click the run folder (e.g. `LIDC/yolo_runs/yolo26s_img1024_e100_b16`) → **Download**. Drive gives you a zip. Keep its name. (Not needed for a brand-new run.)
3. **Training data**, choose one:
   - **Fast:** in Google Drive, right-click `LIDC/yolo_lidc_train_jpg.zip` → Share → General access: **Anyone with the link** → Copy link, and paste it into `DATA_LINK` below. (You can set it back to Restricted afterwards.)
   - **No sharing:** upload `yolo_lidc_train_jpg.zip` into the Kaggle dataset in step 4 instead (about 3 hours from home) and leave `DATA_LINK` empty.
4. **Kaggle → Datasets → New Dataset:** name it `lidc-yolo`, keep it **Private**, and upload the run zip from step 2 and `lidc_code.zip` (from the `colab_upload` folder in the project on the Mac).
5. **Kaggle → Code → New Notebook → File → Import Notebook →** this file.
6. In the notebook's right-hand panel: **Add Input** → Your Datasets → `lidc-yolo`; **Session options**: Accelerator **GPU T4 x2** (the same GPU type as Colab), Internet **On**.
7. **Save Version → Save & Run All (Commit).** It runs in the background, so you can close the browser. Training pauses itself after 11 hours (Kaggle's limit is 12) with a checkpoint that can be continued.

### Next sessions
Open the notebook → **Add Input** → Your Work → this notebook (its latest version's output) → **Save & Run All** again. It always continues the checkpoint with the most epochs.

### Afterwards
The **Output** tab has `yolo_runs/<run>/` with `weights/best.pt`. To go back to Colab, upload that folder to `My Drive/LIDC/yolo_runs/`, replacing the old one.

## 1. Settings

In [ ]:
DATA_LINK = ""                  # Google Drive share link to yolo_lidc_train_jpg.zip, or "" if the zip is in the dataset
MODEL = "yolo26s.pt"            # the model to continue or start, e.g. "yolo11s.pt" for the comparison model
IMGSZ = 1024                    # the image size (same as the earlier runs for a fair comparison)
EPOCHS = 100
BATCH = 16                      # only used when starting a new run (a continued run keeps its own)
STOP_AFTER_HOURS = 11           # pause (resumably) before Kaggle's 12-hour limit

## 2. GPU and packages

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q "ultralytics>=8.4" pydicom gdown

## 3. Code and training data

In [ ]:
import os, shutil, subprocess, zipfile
from pathlib import Path

INPUT, WORK, TMP = Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp/lidc")
DATA_ROOT, RUNS = TMP / "processed", WORK / "yolo_runs"
TMP.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)

def find(pattern):
    return sorted(p for p in INPUT.rglob(pattern) if not p.name.startswith("."))

def show_inputs(depth=4):
    """What Kaggle put in the inputs (folders and files, first few levels)."""
    for p in sorted(INPUT.rglob("*")):
        rel = p.relative_to(INPUT)
        if len(rel.parts) <= depth and not any(x.startswith(".") for x in rel.parts) \
                and not (p.parent.name in ("train", "val") and p.is_file()):
            print("   " * (len(rel.parts) - 1) + p.name + ("/" if p.is_dir() else ""))

# code: lidc_code.zip, or the folder Kaggle unpacked it into
if not (TMP / "code/pipeline/ct.py").exists():
    zips = find("lidc_code.zip")
    cts = [p for p in find("ct.py") if p.parent.name == "pipeline"]
    if zips:
        zipfile.ZipFile(zips[0]).extractall(TMP)
    elif cts:
        shutil.copytree(cts[0].parent.parent, TMP / "code")
    else:
        show_inputs()
        raise SystemExit("lidc_code.zip not found: add the lidc-yolo dataset as an input")
print("code ready")

# training data: the folder Kaggle unpacked the zip into (any name), the zip itself, or the Drive link
if not (DATA_ROOT / "yolo_lidc/val_monitor.txt").exists():
    unpacked = [p.parent for p in find("val_monitor.txt") if (p.parent / "images" / "train").is_dir()]
    zips = [z for z in find("*.zip") if "yolo_lidc_train" in z.name]
    target = DATA_ROOT / "yolo_lidc"
    if unpacked:
        # Kaggle inputs are read-only, and training writes a small dataset_train.yaml into this
        # folder: so make a writable folder that links to the unpacked images and labels
        target.mkdir(parents=True, exist_ok=True)
        for item in unpacked[0].iterdir():
            if not item.name.startswith(".") and not (target / item.name).exists():
                os.symlink(item, target / item.name)
        print(f"training data: using the folder Kaggle unpacked, {unpacked[0]}")
    elif zips or DATA_LINK:
        src = zips[0] if zips else TMP / "data.zip"
        if not zips:
            subprocess.run(["gdown", "--fuzzy", DATA_LINK, "-O", str(src)], check=True)
        zipfile.ZipFile(src).extractall(DATA_ROOT)
        if src.parent == TMP:
            src.unlink()
        print(f"training data: unzipped {src.name}")
    else:
        print("Couldn't find the training data. This is what is in the inputs:")
        show_inputs()
        raise SystemExit("add yolo_lidc_train_jpg.zip to the dataset (and wait until Kaggle has finished "
                         "processing it), or paste the Google Drive link into DATA_LINK in cell 1")

n_train = len(list((DATA_ROOT / "yolo_lidc/images/train").glob("*.*")))
print(f"{n_train} training images (should be 22050)")
assert n_train == 22050

## 4. Find the checkpoint to continue (or start a new run)

Looks through every input (the run zip from Google Drive, earlier versions of this notebook) for runs of `MODEL` at `IMGSZ`, and picks the one with the most finished epochs.

In [ ]:
import pandas as pd
import torch

PREFIX = f"{Path(MODEL).stem}_img{IMGSZ}_e{EPOCHS}_b"     # any batch size

# run folders zipped by Google Drive's download: unpack the ones holding a checkpoint
for z in find("*.zip"):
    with zipfile.ZipFile(z) as zf:
        if any(n.endswith("weights/last.pt") for n in zf.namelist()):
            zf.extractall(TMP / "checkpoints" / z.stem)

def epochs_done(run):
    csv = run / "results.csv"
    return len(pd.read_csv(csv)) if csv.exists() else 0

candidates = [p.parent.parent for p in list(INPUT.rglob("weights/last.pt")) + list((TMP / "checkpoints").rglob("weights/last.pt"))
              if p.parent.parent.name.startswith(PREFIX)]
for c in candidates:
    print(f"  found {c}  ({epochs_done(c)} epochs)")
if candidates:
    best = max(candidates, key=epochs_done)
    run = RUNS / best.name
    if best != run:
        shutil.rmtree(run, ignore_errors=True)
        shutil.copytree(best, run)                # inputs are read-only; continue in /kaggle/working
    ckpt = torch.load(run / "weights/last.pt", map_location="cpu", weights_only=False)
    finished = ckpt.get("optimizer") is None      # YOLO strips the optimizer when a run is complete
    new_run = False
    print(f"\n{'FINISHED' if finished else 'continuing'}: {run.name} after {epochs_done(run)} epochs "
          f"(batch {ckpt['train_args']['batch']}, imgsz {ckpt['train_args']['imgsz']})")
else:
    run, finished, new_run = RUNS / f"{PREFIX}{BATCH}", False, True
    print(f"\nno earlier run of {Path(MODEL).stem} at image size {IMGSZ}: starting a NEW run, {run.name}")

## 5. Train (continue, or start the new run)

Every setting comes from the checkpoint; only the data and output folders are pointed at Kaggle's. Pauses itself after `STOP_AFTER_HOURS`.

In [ ]:
os.environ["LIDC_PROCESSED"] = str(DATA_ROOT)
os.environ["LIDC_YOLO_RUNS"] = str(RUNS)
os.chdir(TMP / "code/pipeline")
WORKERS = os.cpu_count()

if finished:
    print("This run is already finished: download it from the Output tab.")
elif new_run:
    !python 6train_yolo.py --model {MODEL} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --device 0 --workers {WORKERS} --stop-after {STOP_AFTER_HOURS}
else:
    !python 6train_yolo.py --resume "{run}" --device 0 --workers {WORKERS} --stop-after {STOP_AFTER_HOURS}

## 6. Where it got to

In [ ]:
r = pd.read_csv(run / "results.csv")
r.columns = r.columns.str.strip()
print(r[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10).to_string(index=False))
ckpt = torch.load(run / "weights/last.pt", map_location="cpu", weights_only=False)
print("\nFINISHED: download yolo_runs/ from the Output tab" if ckpt.get("optimizer") is None
      else f"\nPAUSED after {epochs_done(run)} epochs: add this version's output as an input and Save & Run All again")